# Mage Iris classifier: runs with outputs (Part C)
Project `iris_project/`, pipeline `iris_classifier`:
`load_iris -> validate_data -> prepare_data -> train_model -> {save_results, evaluate_model}`, each block with its own `@test`.
This notebook runs the **real Mage CLI** from the project root (the cells call `mage run ...`; nothing is simulated) and shows the output.
The accuracy gate in `train_model` is `MIN_ACCURACY = 0.90` (my own choice; the reference notebook was not available).

In [1]:
import os, re, subprocess, sys
from pathlib import Path
ROOT = Path.cwd()
assert (ROOT / "iris_project" / "metadata.yaml").exists(), "run this notebook from the project root"
MAGE = [sys.executable, "-m", "mage_ai.cli.main"] if False else [str(Path(sys.executable).with_name("mage.exe"))]
CSV = ROOT / "iris_predictions.csv"

def mage_run(max_depth=None, show=True):
    """Delete the CSV, run the pipeline with tests (optionally with a max_depth runtime variable), return (exit code, text)."""
    CSV.unlink(missing_ok=True)
    cmd = MAGE + ["run", "iris_project", "iris_classifier", "--test"]
    if max_depth is not None:
        cmd += ["--runtime-vars", '{"max_depth": %d}' % max_depth]
    env = dict(os.environ, PYTHONIOENCODING="utf-8")
    p = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", env=env, cwd=ROOT)
    text = "\n".join(l for l in (p.stdout + p.stderr).splitlines() if l.strip() and "traitlets" not in l and "DBT" not in l and not l.startswith("INFO:"))
    if show:
        print("$ " + " ".join(Path(c).name if i == 0 else c for i, c in enumerate(cmd)))
        print(text)
        print("exit code:", p.returncode, "| iris_predictions.csv created:", CSV.exists())
    return p.returncode, text

## C1: the pipeline with its tests (default `max_depth` = 3)
`mage run iris_project iris_classifier --test`

In [2]:
rc, out = mage_run()

$ mage.exe run iris_project iris_classifier --test
--------------------------------------------------------------
1/1 tests passed.
validate_data: 150 rows, 0 missing values, 3 classes
--------------------------------------------------------------
2/2 tests passed.
--------------------------------------------------------------
1/1 tests passed.
max_depth=3 test_accuracy=1.0000
--------------------------------------------------------------
1/1 tests passed.
Confusion matrix:
                   pred_setosa  pred_versicolor  pred_virginica
actual_setosa               10                0               0
actual_versicolor            0                9               0
actual_virginica             0                0              11
Metrics table:
       class  precision  recall  f1-score  support
      setosa        1.0     1.0       1.0     10.0
  versicolor        1.0     1.0       1.0      9.0
   virginica        1.0     1.0       1.0     11.0
    accuracy        1.0     1.0       1.0     

## C2: tests as a gate: `max_depth` 1 to 5
The CSV is deleted before every run. At depth 1 the `train_model` test fails, so `save_results` and `evaluate_model` must not run and no CSV may appear.

In [3]:
import pandas as pd
rows = []
for d in range(1, 6):
    rc, out = mage_run(d, show=False)
    acc = re.search(r"test_accuracy=([0-9.]+)", out)
    rows.append({"max_depth": d, "test accuracy": float(acc.group(1)) if acc else None,
                 "test_accuracy passed?": "yes" if rc == 0 else "NO", "exit code": rc,
                 "iris_predictions.csv created?": "yes" if CSV.exists() else "no",
                 "save_results ran?": "Saved" in out})
    if d == 1:
        depth1_log = out
pd.DataFrame(rows)

,max_depth,test accuracy,test_accuracy passed?,exit code,iris_predictions.csv created?,save_results ran?
0,1,0.6333,NO,1,no,False
1,2,0.9667,yes,0,yes,True
2,3,1.0000,yes,0,yes,True
3,4,1.0000,yes,0,yes,True
4,5,1.0000,yes,0,yes,True


In [4]:
print("Depth 1 log (the gate in action):")
print(depth1_log)

Depth 1 log (the gate in action):
--------------------------------------------------------------
1/1 tests passed.
validate_data: 150 rows, 0 missing values, 3 classes
--------------------------------------------------------------
2/2 tests passed.
--------------------------------------------------------------
1/1 tests passed.
max_depth=1 test_accuracy=0.6333
FAIL: test_accuracy (block: train_model)
--------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\Ravi\Downloads\DataEngineeringAssignment\freshcart-pipelines\.venv\Lib\site-packages\mage_ai\data_preparation\executors\pipeline_executor.py", line 82, in execute
    loop = asyncio.get_running_loop()
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: no running event loop
During handling of the above exception, another exception occurred:
Traceback (most recent call last):
  File "C:\Users\Ravi\Downloads\DataEngineeringAssignment\freshcart-pipelines\.venv\Lib\site-packages\m

## C3: evaluation branch (confusion matrix and metrics at `max_depth` = 2)

In [5]:
rc, out = mage_run(2, show=False)
start = out.index("Confusion matrix:")
print(out[start:])

Confusion matrix:
                   pred_setosa  pred_versicolor  pred_virginica
actual_setosa               10                0               0
actual_versicolor            0                8               1
actual_virginica             0                0              11
Metrics table:
       class  precision  recall  f1-score  support
      setosa     1.0000  1.0000    1.0000  10.0000
  versicolor     1.0000  0.8889    0.9412   9.0000
   virginica     0.9167  1.0000    0.9565  11.0000
    accuracy     0.9667  0.9667    0.9667   0.9667
   macro avg     0.9722  0.9630    0.9659  30.0000
weighted avg     0.9694  0.9667    0.9664  30.0000
Saved 30 rows to C:\Users\Ravi\Downloads\DataEngineeringAssignment\freshcart-pipelines\iris_predictions.csv
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
Pipeline run completed.


## C4: the extra block `validate_data` and its tests (full run, default depth)

In [6]:
rc, out = mage_run()

$ mage.exe run iris_project iris_classifier --test
--------------------------------------------------------------
1/1 tests passed.
validate_data: 150 rows, 0 missing values, 3 classes
--------------------------------------------------------------
2/2 tests passed.
--------------------------------------------------------------
1/1 tests passed.
max_depth=3 test_accuracy=1.0000
--------------------------------------------------------------
1/1 tests passed.
Confusion matrix:
                   pred_setosa  pred_versicolor  pred_virginica
actual_setosa               10                0               0
actual_versicolor            0                9               0
actual_virginica             0                0              11
Saved 30 rows to C:\Users\Ravi\Downloads\DataEngineeringAssignment\freshcart-pipelines\iris_predictions.csv
Metrics table:
       class  precision  recall  f1-score  support
      setosa        1.0     1.0       1.0     10.0
  versicolor        1.0     1.0       1.

In [7]:
CSV.unlink(missing_ok=True)   # leave no generated file behind (*.csv is git-ignored anyway)
print("graph (metadata.yaml):")
print(Path("iris_project/pipelines/iris_classifier/metadata.yaml").read_text().split("cache_block_output")[0][:1500])

graph (metadata.yaml):
blocks:
- all_upstream_blocks_executed: true
  configuration: {}
  downstream_blocks:
  - validate_data
  executor_type: local_python
  language: python
  name: load_iris
  status: not_executed
  type: data_loader
  upstream_blocks: []
  uuid: load_iris
- all_upstream_blocks_executed: true
  configuration: {}
  downstream_blocks:
  - prepare_data
  executor_type: local_python
  language: python
  name: validate_data
  status: not_executed
  type: transformer
  upstream_blocks:
  - load_iris
  uuid: validate_data
- all_upstream_blocks_executed: true
  configuration: {}
  downstream_blocks:
  - train_model
  executor_type: local_python
  language: python
  name: prepare_data
  status: not_executed
  type: transformer
  upstream_blocks:
  - validate_data
  uuid: prepare_data
- all_upstream_blocks_executed: true
  configuration: {}
  downstream_blocks:
  - save_results
  - evaluate_model
  executor_type: local_python
  language: python
  name: train_model
  status: n